## 1. 문자 단위 RNN

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

### 1) 훈련 데이터 전처리하기
- 문자 시퀀스 apple을 입력 받으면 pple!를 출력하는 RNN 구현
- 입력 데이터와 레이블 데이터에 대한 문자 집합 생성
    - 문자 집합은 중복 제거

In [2]:
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str+label_str)))
vocab_size = len(char_vocab)
print ('문자 집합의 크기 : {}'.format(vocab_size))

문자 집합의 크기 : 5


- 하이퍼파라미터 정의
- 입력은 원-핫 벡터 사용, 입력의 크기=문자 집합의 크기

In [3]:
input_size = vocab_size
hidden_size = 5
output_size = 5
learning_rate = 0.1

- 문자 집합에 고유한 정수 부여

In [4]:
char_to_index = dict((c, i) for i, c in enumerate(char_vocab))
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


- index_to_char: 나중에 예측 결과를 다시 문자 시퀀스로 보기 위함

In [5]:
index_to_char={}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


- 입력 데이터와 레이블 데이터의 각 문자들을 정수로 맵핑

In [6]:
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


- 배치 차원 추가 - nn.RNN()이 기본적으로 3차원 텐서를 입력 받기 때문

In [7]:
#배치 차원 추가
#텐서 연산인 unsqueeze(0)를 통해서도 해결 가능
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


- 입력 시퀀스의 각 문자들을 원-핫 벡터로 변경

In [8]:
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


- 입력 데이터와 레이블 데이터를 텐서로 변경

In [9]:
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

/tmp/ipython-input-2348034151.py:1: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  X = torch.FloatTensor(x_one_hot)


In [10]:
print('훈련 데이터의 크기: {}'.format(X.shape))
print('레이블의 크기: {}'.format(Y.shape))

훈련 데이터의 크기: torch.Size([1, 5, 5])
레이블의 크기: torch.Size([1, 5])


### 2) 모델 구현하기

In [11]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True) #RNN 셀 구현
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True) #출력층 구현

    def forward(self, x): # 구현한 RNN 셀과 출력층을 연결
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

- 클래스로 정의한 모델을 net에 저장

In [12]:
net = Net(input_size, hidden_size, output_size)

- 입력된 모델에 입력을 넣어서 출력의 크기를 확인

In [13]:
outputs = net(X)
print(outputs.shape) #3차원 텐서

torch.Size([1, 5, 5])


- view를 사용하여 배치 차원과 시점 차원을 하나로 만듦

In [14]:
print(outputs.view(-1, input_size).shape) #2차원 텐서로 변환

torch.Size([5, 5])


In [15]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


- 옵티마이저와 손실 함수 정의

In [16]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

- 100 에포크 학습

In [17]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1)) #Batch 차원 제거를 위해 view를 함
    loss.backward() #기울기 계산
    optimizer.step() #optimizer 선언 시 넣어둔 파라미터 업데이트

    result = outputs.data.numpy().argmax(axis=2) #최종 예측값인 각 time-step 별 5차원 벡터에 대해서 가장 높은 값의 인덱스를 선택
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss: ", loss.item(), "prediction: ", result, "true Y: ", y_data, "prediction str: ", result_str)

0 loss:  1.6773383617401123 prediction:  [[2 2 2 2 2]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  eeeee
1 loss:  1.4413706064224243 prediction:  [[2 3 3 2 2]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  ellee
2 loss:  1.259540319442749 prediction:  [[2 3 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  elle!
3 loss:  1.0795100927352905 prediction:  [[2 3 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  elle!
4 loss:  0.892647922039032 prediction:  [[2 3 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  elle!
5 loss:  0.7280318140983582 prediction:  [[4 0 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  p!le!
6 loss:  0.5821952223777771 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
7 loss:  0.4572128355503082 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
8 loss:  0.3554021716117859 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
9 loss:  0.27375465631484985 prediction:  [[4 4 3 2 0]] t

## 더 많은 데이터로 학습한 문자 단위 RNN

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim

### 1) 훈련 데이터 전처리하기

In [19]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

- 문자 집합 생성, 각 문자에 고유한 정수 부여

In [20]:
char_set = list(set(sentence)) #중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} #각 문자에 정수 인코딩

In [21]:
print(char_dic)

{"'": 0, 'f': 1, 't': 2, 's': 3, ',': 4, 'd': 5, 'i': 6, 'o': 7, '.': 8, 'w': 9, 'm': 10, ' ': 11, 'k': 12, 'h': 13, 'b': 14, 'u': 15, 'e': 16, 'g': 17, 'p': 18, 'l': 19, 'y': 20, 'a': 21, 'c': 22, 'n': 23, 'r': 24}


In [22]:
dic_size = len(char_dic)
print('문자 집합의 크기: {}'.format(dic_size))

문자 집합의 크기: 25


- 하이퍼파라미터 설정
- sequence_length 변수 선언

In [23]:
#하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10  #임의 숫자 지정
learning_rate = 0.1

- 임의로 지정한 sequence_length 값인 10의 단위로 샘플들을 잘라서 데이터를 만듦

In [24]:
#데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  #x str to index
    y_data.append([char_dic[c] for c in y_str])  #y str to index

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

In [25]:
print(x_data[0])
print(y_data[0])

[6, 1, 11, 20, 7, 15, 11, 9, 21, 23]
[1, 11, 20, 7, 15, 11, 9, 21, 23, 2]


- 입력 시퀀스에 대해서 원-핫 인코딩 수행, 입력 데이터와 레이블 데이터를 텐서로 변환

In [26]:
x_one_hot = [np.eye(dic_size)[x] for x in x_data] #x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [27]:
print('훈련 데이터의 크기: {}'.format(X.shape))
print('레이블의 크기: {}'.format(Y.shape))

훈련 데이터의 크기: torch.Size([170, 10, 25])
레이블의 크기: torch.Size([170, 10])


In [28]:
print(X[0])

tensor([[0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [29]:
print(Y[0])

tensor([ 1, 11, 20,  7, 15, 11,  9, 21, 23,  2])


### 2) 모델 구현하기
- 은닉층 2개 쌓기

In [30]:
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers): #현재 hidden_size는 dic_size와 같음
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [31]:
net = Net(dic_size, hidden_size, 2) #이번에는 층을 두 개 쌓음

- nn.RNN() 안에 num_layers라는 인자 사용, 은닉층을 몇 개 쌓을 것인지를 의미
- 비용 함수와 옵티마이저 선언

In [32]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

- 출력 크기 확인

In [33]:
outputs = net(X)
print(outputs.shape) #3차원 텐서

torch.Size([170, 10, 25])


- view를 사용하여 배치 차원과 시점 차원을 하나로 만듦

In [34]:
print(outputs.view(-1, dic_size).shape) #2차원 텐서로 변환

torch.Size([1700, 25])


In [35]:
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([170, 10])
torch.Size([1700])


- 학습

In [36]:
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X) #(170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    #results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: #처음에는 예측 결과를 전부 가져옴
            predict_str += ''.join([char_set[t] for t in result])
        else: #그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)

' afh'f hhf    a'h af hfhhfff ' a   fhff'fhff'f f,f hff hfff  h'   hf  h'hf hhf hha   afhmff  hyff afff hhf hf'ff '  ahf hfff  hohf hyff   'hhffhfhf hffhh   ff  ffhff  fhfhf hff' 
o                                                                                                                                                                                  
  tt  tt      t    t tt    tt t    tt  t  t   tt t  tt  tt  ttt  t  tt   tt  t  ttt   t tt  t  t   t   tt t   t   t  t  t     t   t  tt  t   tt tt   tt  tt   t t  tttt  ttt   t  t
tb.  a.a aa  ab.  b  aa    aa a a aaa aa  ba  bb aa ba abb abb  bb  ab  aba aa  aba  aa aaaaa aaaaab  aaa a aab  aa aa aa. aabu  ba  aa ab  aau ab  abf abb aaaaaaaaaab aaaaa aa  b
ehe e eah hehdoeaheho,ahemo oeah ho hhemo oehh ho hhodooah hoooahemo ao osah hosooohh hhgshehho moahoho heg oahah ah oeahhoahah aohhoomh moao me ooaoaoooohoh hhh o hgheoohh eoshho
e e e e e      e     h                           h                      h ue      e                 